# **Bishop - 2024**

## **13 Graph neural networks**

### 13.1 **Machihe learning on graphs**

#### 13.1.1 **Graph properties**

- **$X$:** is the data matrix, where each line represents **$x_n$**, the information of each node
- **$A$:** is the adjacency matrix of the Graph

#### 13.1.3   **Permutation equivariance**

If **$\tilde{X} = PX$** and **$\tilde{A} = PAP^T$** 

- We wish to have two properties:
    - **$y(\tilde{X},\tilde{A}) = y(X,A)$**  **Invariance**
    - **$y(\tilde{X},\tilde{A}) = Py(X,A)$**  **Equivariance** 

### 13.2 **Neural message-passing**

Ensuring invariance or equivarence under node label permutation is a key design consideration. We may also wish to retain the concept of "layer", in order to exploit the deep neural networks capabilities. If each layer of the network is equivariant under node representations, then multiple layers will also exhibit equivariance, while allowing each layer of the neural network to be informed by the graph structure. 

We also want to ensure that each layer is a high flexible nonlinear function and is differentiable with respect to its paramenter so that it can be trained by stochastic gradient descent with respect to its paramenter so that it can be trained.

A further requirement is that the network should be able to handle different lengths of inputs.

#### 13.2.1 **Convolutional filters**

We can seek inspiration from image processinf using convolutional neural networks. If we map each pixel of an image to an graph, then the neighbours of an pixel would be the other pixels adjacency to it on the image. When performing an convolution throught the layers, we do the following computaion:
**$$z_i^{(l + 1)} = f \left( \sum_{j}{ w_j z_j^{(l)} + b}\right)$$**
where the sum over **$j$** is taken over all nine pixels in a small patch in layer **$l$**. The same function **$f$**, which is a differentiable nonlinear activation function, is applied across multiple patches in the image, so that the weights **$w_j$** and bias **$b$** are shared across patches.

As it stands, the computation of the next layer node is not invariant under permutatins of the pixels in the precious nodes, because the weights are multiplied by the nodes on a specific place in the previous layer. However, we can aachieve this by representing the filter as a graph, where all the neighbours of **$i$**, **$\mathcal{N(i)}$** share a single parameter **$w_{neigh}$** so that

**$$z_i^{(l + 1)} = f \left( w_{neigh} \sum_{j \in \mathcal{N(i)}}{ z_j^{(l)}} + w_{self} z_i^{(l)}  + b\right)$$**

where node **$i$** has its own weight parameter **$w_{self}$**

We can interpret the formula as updating a local representation *$$z_i$$* as node **$i$** by gathering information from the neighbours nodes by passing *massages* from neighbours node into node $i$.

#### 13.2.2 **Graph convolutional networks**

For each node $n$ in the graph and for each layer $l$ in the network, we intoduce a $D$-dimensional conlumn vector **$h_n^{(l)}$** of node-embedding variables.

We can view each layer of processing as having two successive stages. The first is the *agregation* stage in which, for each node $n$, massages are passed to that node from its neighbours and combined to form a new vector **$z_n^{(l)}$** in a way that is permutation invariant. This is followed by an *update* step in which the aggregated information from neighbouring nodes is combined with local information from the node itself and used to calculate a revised embedding vector for that node.

**$$z_n^{(l)} = \text{Aggregate}\left( \set{h_m^{(l)} : m \in \mathcal{N(i)}} \right) $$**
**$$h_n^{(l + 1)} = \text{Update} \left( h_n^{(l)}, z_n^{(l)} \right)$$**

Both this operation can be a function of a set of learnable parameters, as long as it is differentiable. This frame work is called a *message-passing neural network*


#### 13.2.3 **Aggregation operators**

There are many forms for the Aggregate function, but it must depend only on the set of inputs and not on their ordering. It must also be a differentiable funcion of any learnable parameters.

**$$z_n^{(l)} = \text{Aggregate}\left( \set{h_m^{(l)} : m \in \mathcal{N(n)}} \right) = \sum_{m \in \mathcal{N(n)}}{\frac{h_m^{(l)}}{\sqrt{|\mathcal{N(n)}||\mathcal{N(m)}|}}}$$**

is one example of aggragate function


As information is passed through the layers, the updates to a given node depend on a steadily increasing fraction of other nodes in earlier layers until the effective receptive field potentially spans the whole graph. However, large, sparse graphs may require an excessive number of layers before each output is influenced by every input. Some archtectures therefore introduce an additional 'super-node' that connects to every node in the original graph to ensure fast propagation of information.

We can also use an $MLP$ 'multilayer perceptron' to cumpute the aggregation, in order to obtain treinable parameters.

**$$z_n^{(l)} = \text{Aggregate}\left( \set{h_m^{(l)} : m \in \mathcal{N(n)}} \right) = MLP_{\theta} \left(\sum_{m \in \mathcal{N(n)}} {MLP_{\phi}({h_m^{(l)}})}\right)$$**

where $MLP_{\theta}$ represents an multilayer perceptron with parameters $\theta$

#### 13.2.4 **Update operators**

By analogy with the CNN, a simple form of this operator should be

**$$h_n^{(l + 1)} = \text{Update} \left( h_n^{(l)}, z_n^{(l)} \right) = f(W_{self} h_n^{(l)} + W_{neigh} z_n^{(l)} + b)$$**

Overall, we can represent a graph neural network as a sequence of layers that successively transform the node embeddings. If we group these embeddings into a matrix **$H$** whose $n_{th}$ row is the vector **$h_n^T$** , which is initialized to the data matrix **$X$**

**$$H^{(1)} = F(X, A, W^{(1)}) $$** **$$H^{(2)} = F(X, A, W^{(2)})$$** **$$\vdots$$** **$$H^{(L)} = F(H^{(L - 1)}, A, W^{(L)})$$**

where **$A$** is the adjacency matrix, and **$W^{(l)}$** represents the complete set of weight and biases in layer $l$ of the network.

#### 13.2.5 **Node classification**